# Lesson 12 Lab — Sparse Regularization and Learnable Structural Gates

**Puzzle:** Can training produce a reproducible structural ranking instead of choosing a threshold after the fact?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Learnable gates attach a continuous variable to each channel and optimize it with the task. A sparsity penalty can separate useful and dispensable structures, but thresholding still creates a discrete architecture and must be evaluated. Gate values, penalty strength, temperature, and threshold belong in the artifact.


## 0. Predict before running

1. Predict whether sigmoid gates reach exact zero under an L1 penalty.
2. Predict how increasing lambda changes active channels and task loss.
3. Choose a threshold using a frozen validation objective rather than the training batch.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A frozen feature tensor, a trainable gated linear predictor, per-channel sigmoid gates, a task loss, an L1 gate penalty, and a threshold sweep form the lab.

- Regularization shapes a ranking but does not physically remove channels.
- Continuous and thresholded objectives must both be reported.
- Lambda and threshold define different parts of the sparsity trade-off.


## 2. Derive the mechanism

With gate `g_c=sigmoid(a_c)`, a hidden feature becomes `g_c h_c`. Optimizing `L_task + lambda sum(g_c)` trades fit against active width. Sigmoid gates rarely become exact zeros, so deployment selects a threshold or top-k budget and physically rebuilds the layer. The continuous optimum and discrete candidate are different models; both losses must be measured. Gate scale can also trade with neighboring weights unless those degrees of freedom are controlled.

### Mechanism at a glance

```mermaid
flowchart LR
  X["activation"] --> G["learnable structural gate"]
  T["task loss"] --> O["joint optimization"]
  R["sparsity regularizer"] --> O
  O --> G
  G --> H["threshold and freeze indices"]
  H --> P["physical graph surgery"]
  P --> V["recover + validate"]
```

### Walk it step by step

1. **Attach a learnable gate.** Place one gate on the structural unit to be selected, such as a channel, head, or block.
2. **Optimize task and sparsity objectives together.** Track the task loss, regularization pressure, and gate distribution rather than only the final zero count.
3. **Freeze a discrete structure.** Choose and record a threshold, then convert soft gates into an explicit retained-index set.
4. **Remove the gated structure physically.** Rebuild and recover the model so the runtime sees smaller tensors instead of multiplying by near-zero gates.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 12
LESSON_TITLE = 'Sparse Regularization and Learnable Structural Gates'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260820
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | weak gate regularization with a mostly dense effective width |
| Candidate | stronger regularization plus discrete threshold candidates |
| Held constant | features, targets, predictor initialization, optimizer steps, thresholds, seed, and validation split |
| Measurements | gate distribution, active channels, continuous loss, thresholded loss, and selected threshold |
| Evidence | `numerical-model` |

**Experiment:** Train channel gates under two regularization strengths and evaluate a frozen threshold sweep.


## 5. Read the experiment code

The feature generator deliberately makes only a subset of channels predictive. Two gate models begin from identical logits, and the threshold sweep evaluates held-out loss without further fitting. This isolates whether the learned ranking exposes the known sparse structure.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
n,d=1600,24
x=torch.randn(n,d,device=DEVICE); true_w=torch.zeros(d,device=DEVICE); true_w[:6]=torch.tensor([2.0,-1.5,1.2,-0.9,0.7,0.5],device=DEVICE); y=x@true_w+0.15*torch.randn(n,device=DEVICE)
tx,vx=x[:1200],x[1200:]; ty,vy=y[:1200],y[1200:]
class Gated(nn.Module):
    def __init__(self): super().__init__(); self.logits=nn.Parameter(torch.zeros(d)); self.weight=nn.Parameter(torch.randn(d)*0.05)
    def gates(self): return torch.sigmoid(self.logits)
    def forward(self,z): return (z*self.gates())@self.weight
def train_gate(lam):
    m=Gated().to(DEVICE); opt=torch.optim.Adam(m.parameters(),lr=0.04)
    for step in range(260):
        idx=torch.arange(step*96,step*96+96,device=DEVICE)%tx.shape[0]; opt.zero_grad(); pred=m(tx[idx]); loss=F.mse_loss(pred,ty[idx])+lam*m.gates().mean(); loss.backward(); opt.step()
    return m
weak=train_gate(0.005); strong=train_gate(0.15)
thresholds=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
rows=[]
with torch.inference_mode():
    for th in thresholds:
        g=strong.gates(); mask=(g>=th).float(); pred=(vx*g*mask)@strong.weight; rows.append({"threshold":th,"active":int(mask.sum().item()),"mse":float(F.mse_loss(pred,vy).item())})
best=min(rows,key=lambda r:r["mse"]); wg=weak.gates().detach(); sg=strong.gates().detach()
metrics={"weak_active_channels":int((wg>=0.5).sum().item()),"strong_active_channels":int((sg>=0.5).sum().item()),"weak_gate_mean":float(wg.mean().item()),"strong_gate_mean":float(sg.mean().item()),"weak_val_mse":float(F.mse_loss(weak(vx),vy).item()),"strong_val_mse":float(F.mse_loss(strong(vx),vy).item()),"selected_threshold":best["threshold"],"thresholded_val_mse":best["mse"],"threshold_sweep":rows,"strong_gates":sg.cpu().tolist()}
analysis=(f"Weak regularization left {metrics['weak_active_channels']} gates above 0.5 with mean {metrics['weak_gate_mean']:.4f}; "
          f"strong regularization left {metrics['strong_active_channels']} with mean {metrics['strong_gate_mean']:.4f}. "
          f"The frozen threshold sweep selected {best['threshold']:.1f}, {best['active']} active channels, and validation MSE {best['mse']:.6f}. Continuous gates still require physical rebuilding.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Weak active channels | 6 |
| Strong active channels | 6 |
| Weak gate mean | 0.468424 |
| Strong gate mean | 0.259656 |
| Selected threshold | 0.200000 |
| Thresholded validation MSE | 0.019543 |


## 7. Interpret rather than merely print

Weak regularization left 6 gates above 0.5 with mean 0.4684; strong regularization left 6 with mean 0.2597. The frozen threshold sweep selected 0.2, 6 active channels, and validation MSE 0.019543. Continuous gates still require physical rebuilding.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 12,
    "title": 'Sparse Regularization and Learnable Structural Gates',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Learnable gates turn structure selection into optimization, but deployment still requires a separately validated discrete architecture.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 12,
  "title": "Sparse Regularization and Learnable Structural Gates",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260820
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "weak_active_channels": 6,
    "strong_active_channels": 6,
    "weak_gate_mean": 0.4684237241744995,
    "strong_gate_mean": 0.2596563696861267,
    "weak_val_mse": 0.020268043503165245,
    "strong_val_mse": 0.020011305809020996,
    "selected_threshold": 0.2,
    "thresholded_val_mse": 0.019542662426829338,
    "threshold_sweep": [
      {
        "threshold": 0.1,
        "active": 13,
        "mse": 0.019915562123060226
      },
      {
        "threshold": 0.2,
        "active": 6,
        "mse": 0.019542662426829338
      },
      {
        "threshold": 0.3,
        "active": 6,
        "mse": 0.019542662426829338
      },
      {
        "threshold

## 9. Make the bounded decision

> Learnable gates turn structure selection into optimization, but deployment still requires a separately validated discrete architecture.

**Acceptance/rollback:** Accept a gate-derived architecture only when threshold selection is frozen, held-out quality passes, and the physical model reproduces the gated candidate.

**Failure analysis:** Jointly trainable downstream weights can absorb inverse gate scaling, making raw gates misleading. Selecting lambda or threshold on the final test set leaks evaluation. Hard thresholding may also remove interacting channels that looked individually small.


## 10. Extend the evidence

Add hard-concrete or top-k gates, rebuild a physical narrow layer, and test ranking stability across seeds and task slices.

The full evidence boundary and references are in [`README.md`](README.md).
